# 🛡️ Detector UEBA: Detecção de Anomalias em Logs de Autenticação
### Dissertação de Mestrado Profissional em Engenharia Elétrica (UnB / FT / ENE)
* **Autor:** Allysson de Oliveira Noronha
* **Orientador:** Prof. Dr. Robson de Oliveira Albuquerque
* **Repositório:** [https://github.com/allyssonnor/anti_Detector](https://github.com/allyssonnor/anti_Detector)

---

### 🎯 Propósito deste Demonstrador
Este notebook foi desenvolvido para que a **Banca Examinadora** possa reproduzir e auditar a comprovação da hipótese central da dissertação:
> **Hipótese:** Modelos Especialistas em domínios comportamentais específicos (ex: autenticação/força bruta) superam modelos Generalistas em fidelidade, mitigando alertas espúrios enquanto retêm altíssimo recall nas ameaças-alvo.

### 🔬 Estrutura da Avaliação Científica:
1. **Auditoria de Causalidade (`pytest`):** Garantia matemática e algorítmica de ausência de vazamento temporal (*data leakage*).
2. **Treinamento da Matriz Experimental (16 Modelos):**
   * **Generalistas (`EF_Amplo`):** Treinados sobre expediente normal com ataques multimodais simultâneos (Força Bruta, Fora de Hora, Movimentação Lateral).
   * **Especialistas (`EF_Bruteforce`):** Treinados sobre expediente normal e exclusivamente campanhas de Força Bruta e Password Spraying.
   * **Variações:** Particionamento Cronológico vs Aleatório, Limiares Globais vs Por Usuário, Random Forest e MLP Autoencoder.
3. **Inferência Cruzada Multicenário (*Zero-Shot Domain Transfer*):**
   * Generalistas inferindo contra `seed_1_sample`, `ef_bruteforce` e `test_bruteforce_42`.
   * Especialistas inferindo contra `test_bruteforce_42`, `ef_amplo` e `seed_1_sample`.
4. **Dashboard Forense e Métricas Consolidadas:** Comparação direta de TP, FP, FN, TN, Precisão, Recall, F1, F2 e PR-AUC.

## 1. ⚙️ Preparação do Ambiente e Clonagem do Repositório

In [ ]:
# 1. Clonagem resiliente do repositório (diretório fixo /content/anti_Detector)
import os

%cd /content
if not os.path.isdir("/content/anti_Detector"):
    !git clone https://github.com/allyssonnor/anti_Detector.git
%cd /content/anti_Detector
!git pull

# 2. Instalação das dependências necessárias
!pip install -q -r requirements.txt


## 2. 🧪 Auditoria de Integridade e Causalidade Temporal
Execução da suíte de testes de causalidade para comprovar que nenhuma feature utiliza informação futura no instante $t$.

In [ ]:
!python -m pytest tests/test_causality.py -v

## 3. 🏋️ Treinamento da Matriz de Modelos (Generalistas vs Especialistas)
Execução do treinamento cruzando:
* **Datasets:** `EF_Amplo` (Generalista) e `EF_Bruteforce` (Especialista)
* **Splits:** `chronological` e `random`
* **Modos de Limiar:** `global` e `per_user` (calibrados dinamicamente via $F_2$-Score na validação)
* **Algoritmos:** `RandomForest` (supervisionado) e `MLP_Single` (Autoencoder neural $X \to X$)

In [ ]:
## 4. 📊 Tabela Consolidada de Resultados de Validação e Teste Misto

import pandas as pd
import glob

# Busca resiliente pelo consolidado master_results_table.csv
res_csv = glob.glob("**/master_results_table.csv", recursive=True)
if not res_csv:
    res_csv = glob.glob("results/**/master_results*.csv", recursive=True)

if res_csv:
    df_res = pd.read_csv(res_csv[0])
    cols = [c for c in ["dataset", "model_preset", "sampling.split_strategy", "training.threshold_mode", "precision", "recall", "f1_score", "f2_score", "pr_auc", "roc_auc", "lift_pr", "tp", "fp", "fn", "tn"] if c in df_res.columns]
    print("\n📈 TABELA CONSOLIDADA DE VALIDAÇÃO (16 Modelos Avaliados):")
    display(df_res[cols].drop_duplicates().sort_values(by=["dataset", "model_preset"]))
else:
    print("Nenhum resultado consolidado encontrado. Verifique se o treino na Célula 3 concluiu.")


In [ ]:
## 5. 🔍 Bateria de Inferência Cruzada (Comprovação da Especialização)
Execução automatizada dos 6 cenários de inferência cruzada para comparar o comportamento dos Generalistas versus Especialistas:

| Grupo | Modelo | Dataset Alvo | Comportamento Esperado |
| :--- | :--- | :--- | :--- |
| **Generalista** | `EF_Amplo (RF)` | `seed_1_sample` | Detecção distribuída sobre múltiplos vetores de ameaça. |
| **Generalista** | `EF_Amplo (RF)` | `ef_bruteforce` | Perda de precisão específica por diluição dos pesos entre múltiplos tipos de ataque. |
| **Generalista** | `EF_Amplo (RF)` | `test_bruteforce_42` | Menor contraste discriminativo frente a rajadas focadas de autenticação. |
| **Especialista** | `EF_Bruteforce (RF)` | `test_bruteforce_42` | **Altíssimo Recall e Precisão:** domínio focado no vetor de força bruta trivial. |
| **Especialista** | `EF_Bruteforce (RF)` | `ef_amplo` | **Seletividade:** detecta a força bruta e deixa passar os outros ataques (FNs desejados). |
| **Especialista** | `EF_Bruteforce (RF)` | `seed_1_sample` | **Seletividade:** captura os ataques de força bruta no cenário massivo ignorando outros desvios. |

# Execução da Bateria Completa de Inferência Cruzada
!python scripts/run_cross_inference.py


In [ ]:
## 6. 🎨 Dashboards Forenses Comparativos e Explicabilidade
Visualização lado a lado da Matriz de Confusão e das métricas operacionais para o Especialista no ataque trivial (`test_bruteforce_42`) versus nos cenários mistos.

from IPython.display import Image, display
import glob
import os
import pandas as pd

# Exibição do Dashboard do Especialista contra o Ataque de Bruteforce 42
esp_dir = "results/demo_colab/full/run_seed42/EF_Bruteforce_Chrono_RandomForest_global_20"
dash_42 = f"{esp_dir}/dashboard_test_bruteforce_42.png"
if os.path.exists(dash_42):
    print("\n🎯 1. Desempenho do Especialista contra o Ataque Isolado (test_bruteforce_42):")
    display(Image(dash_42))

# Exibição do Dashboard do Especialista contra o Cenário Amplo (Seletividade)
dash_amplo = f"{esp_dir}/dashboard_ef_amplo.png"
if os.path.exists(dash_amplo):
    print("\n🛡️ 2. Seletividade do Especialista contra o Cenário Multimodal (ef_amplo):")
    display(Image(dash_amplo))

# Amostra da Tabela Consolidada de Inferência Cruzada
csv_summary = "results/demo_colab/tabela_inferencia_cruzada.csv"
if os.path.exists(csv_summary):
    df_summary = pd.read_csv(csv_summary)
    print("\n📊 Tabela Comparativa de Generalização (Zero-Shot Domain Transfer):")
    display(df_summary)


In [ ]:
from IPython.display import Image, display
import glob
import os
import pandas as pd

# Busca pelo dashboard gerado na inferência mais recente
dashboards = glob.glob(f"{MODEL_DIR}/dashboard_*.png")
if dashboards:
    display(Image(dashboards[0]))
else:
    print(f"Dashboard não encontrado em: {MODEL_DIR}")

# Amostra das detecções e anomalias identificadas
eventos_csvs = glob.glob(f"{MODEL_DIR}/inferencia_*.csv")
if eventos_csvs:
    df_inf = pd.read_csv(eventos_csvs[0])
    print("\n🛡️ Top 10 Alertas Disparados com Detalhes Comportamentais:")
    alertas = df_inf[df_inf["alert"] == 1] if "alert" in df_inf.columns else df_inf.head(10)
    cols_show = [c for c in ["UserID", "Time", "IpAddress", "LogHost", "anomaly_score", "threshold", "alert", "Is_Anomaly", "AnomalyType"] if c in df_inf.columns]
    display(alertas[cols_show].head(10))
